In [ ]:
import pandas as pd
from pathlib import Path
import sys

import os

# Get the current working directory
_script_dir = Path(os.getcwd())

# The rest of your logic
_project_root = _script_dir.parent.parent
if str(_project_root) not in sys.path:
    sys.path.append(str(_project_root))

from src.config import (
    RAW_GAMES_PATH,
    REGULAR_SEASON_GAMES_PATH,
    TEAMS_CITIES_CONFERENCE_HISTORY_PROCESSED_PATH,
    GAMES_FEATURES_PATH,
)

from src.etl.ingestion import (
    build_regular_season_games,
    load_teams_history_table,
)
from src.etl.transformation import add_conference
from src.etl.features import create_features_tables, merge_features


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
raw_games_path: Path = None
teams_history_path: Path = None
output_path: Path = None

In [ ]:
# Step 1: Load teams history table
print("\n[Step 1/5] Loading teams history table...")
if teams_history_path is None:
    teams_history_path = TEAMS_CITIES_CONFERENCE_HISTORY_PROCESSED_PATH

teams_history = load_teams_history_table(processed_file=str(teams_history_path))
print(f"✓ Loaded teams history table with {len(teams_history)} rows")


[Step 1/5] Creating teams history table...
Output saved to /Users/felipeformenti/dev/fformenti/nba_bets/data/processed/teams_history_expanded.csv
✓ Created teams history table with 1536 rows


In [ ]:
# Step 2: Parse and filter raw games
print("\n[Step 2/5] Parsing and filtering raw games...")
if raw_games_path is None:
    raw_games_path = RAW_GAMES_PATH

raw_games = pd.read_csv(raw_games_path, parse_dates=["gameDate"], low_memory=False)
regular_season_games = build_regular_season_games(raw_games)
regular_season_games.to_csv(REGULAR_SEASON_GAMES_PATH, index=False)
print(f"✓ Processed {len(regular_season_games)} regular season games")

# Step 3: Add conference information
print("\n[Step 3/5] Adding conference information...")
games_with_conference = add_conference(regular_season_games, teams_history)
print(f"✓ Added conference information to {len(games_with_conference)} games")



[Step 2/5] Parsing and filtering raw games...


/var/folders/_2/8lt451812jdgdwl3yhp1bbhh0000gn/T/ipykernel_8093/2604310280.py:6: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_games = pd.read_csv(raw_games_path, parse_dates=["gameDate"])


✓ Processed 65498 regular season games

[Step 3/5] Adding conference information...
✓ Added conference information to 65498 games


In [6]:
# games_with_conference_1980 = games_with_conference[
#     games_with_conference["gameDate"] > "1980-01-01"
# ]
# games_with_conference_1980_no_conference = games_with_conference_1980[
#     pd.isna(games_with_conference_1980["hometeamConference"])
# ]

# cols = [
#     "hometeamId",
#     "hometeamName",
#     "hometeamCity",
#     "season",
# ]
# games_with_conference_1980_no_conference[cols].drop_duplicates()

In [ ]:
# Step 4: Create feature tables
print("\n[Step 4/5] Creating feature tables...")
lags = [10, 82]
location_lags = [5, 41]
create_features_tables(
    games_with_conference, lags=lags, location_lags=location_lags
)
print("✓ Created all feature tables")

# Step 5: Merge features
print("\n[Step 5/5] Merging features into final table...")
if output_path is None:
    output_path = GAMES_FEATURES_PATH

final_features = merge_features(games_with_conference)
final_features.to_csv(output_path, index=False)
print(f"✓ Created final features table with {len(final_features)} rows")
print(f"✓ Saved to: {output_path}")

print("\n" + "=" * 60)
print("Pipeline completed successfully!")
print("=" * 60)

In [ ]:
# final_features_filtered = final_features[final_features["gameDate"] > "1980-01-01"]
final_features_filtered = final_features.loc[
    (final_features["season"] == "1997/98") & 
    ((final_features["hometeamCity"] == "Boston") | 
    (final_features["awayteamCity"] == "Boston"))
    ]

# final_features_filtered = final_features.loc[
#     (final_features["season"] == "1997/98") & (final_features["hometeamCity"] == "Boston")
# ]

In [ ]:
final_features_filtered


In [ ]:
# final_features_filtered[
#     [
#         "gameDate",
#         "hometeamName",
#         "awayteamName",
#         "rested_days_HT",
#         "rested_days_VT",
#         "days_at_home",
#         "days_on_road",
#     ]
# ].sort_values(by="gameDate", ascending=True)


In [ ]:
# final_features_filtered.drop(columns=["gameId", "hometeamId", "awayteamId", "hometeamName", "awayteamName", "gameType", "gameDateOnlyStr"]).sort_values(by="gameDate", ascending=True)[0:10]


In [ ]:
# final_features_filtered[final_features_filtered["east_wins_pct_L1"].isna()]


In [ ]:
# final_features.columns


In [ ]:
# mask = (
#     (final_features["hometeamId"] == 1610612738)
#     | (final_features["awayteamId"] == 1610612738)
# ) & (final_features["season"] == "1996/97")
# final_features.loc[
#     mask,
#     [
#         "gameDate",
#         "hometeamName",
#         "awayteamName",
#         "homeScore",
#         "awayScore",
#         "win_bool",
#         "pts_diff",
#         "record_HT",
#         "record_VT",
#         "pts_diff_avg_HT",
#         "pts_diff_avg_VT",
#         "east_wins_pct_L1",
#     ],
# ]


In [ ]:
# final_features["win_bool"].mean()


In [ ]:
# final_features[
#     (pd.isna(final_features["east_wins_pct_L1"]))
#     & (final_features["gameDate"] > "1990-01-01")
# ]


In [ ]:
final_features[final_features["gameDateOnlyStr"] == "2002-04-15"]

In [ ]:
# final_features


In [ ]:
debug_season = final_features.loc[
    final_features["season"] == "2001/02",
    [
        "gameDate",
        "hometeamName",
        "awayteamName",
        "games_played_HT",
        "games_played_VT",
        "pts_diff",
        "winnerteamConference",
        "east_wins_pct_L1",
    ],
]

In [ ]:
debug_season.sort_values(by="gameDate", ascending=True)[0:100]

In [ ]:
a = set()
b = set([0])


a.difference(b)

In [ ]:
total_games = 10
east_home_games = 6
west_home_games = total_games - east_home_games
east_record = 0.6
west_record = 1 - east_record

print("east:", east_record * 5 / (east_home_games))
print("west:", west_record * 5 / (west_home_games))